# ISI-1 Non-Monotonicity Investigation

When fitting `sigma1`, d'(ISI=1) vs sigma1 shows a **non-monotonic** bump —
performance *increases* before decreasing — unlike ISI=2 or ISI=4.

This notebook systematically investigates possible causes:

| Investigation | Variable | Goal |
|---|---|---|
| A | sigma0 | Does the bump disappear at certain sigma0 values? |
| B | t_step | Does changing the regime boundary (T=4,5,6) matter? |
| C | ISI distribution | Does using ISI=[1]-only vs ISI=[1,2,4] mixed sequences change the bump? |
| D | Score distributions | What happens to hit/FA distributions at the bump? |

In [1]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from scipy.spatial.distance import pdist

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.plotting import ensure_dir
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.io_utils import (
    make_model_save_dir,
    save_all_figures,
    save_single_figure,
    save_runs_summary,
)
from encoders import *

from utls.toy_experiments import (
    make_toy_experiment_list,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
)
from utls.sigma_fitting import (
    log_mid,
    fit_sigma_1d,
    plot_sigma_fit,
)

## 1. Load config & data

In [2]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path


CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)

model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

{'results_root': '/om2/user/bjmedina/auditory-memory/memory', 'tag': 'slurm', 'experiment': {'is_multi': True, 'n_seqs': 36, 'n_samples': 50, 'which_task': 0}, 'metric': 'cosine', 'noise_model': {'name': 'three-regime', 'sigma0_min': 3.0, 'sigma0_max': 0.5, 'sigma1_min': 0.1, 'sigma1_max': 0.6, 'sigma2_min': 0.0005, 'sigma2_max': 0.1, 't_step': 5}, 'run_id': 'run_000005', 'representation': {'type': 'resnet50', 'layer': 'layer4', 'time_avg': False}}


In [3]:
# ---- experiment ----
exp_cfg = model_cfg["experiment"]
which_task = exp_cfg["which_task"]
is_multi = exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)

isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
t_step = noise_cfg["t_step"]

# ---- representation ----
repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

# ---- load human data ----
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)

human_curve = compute_human_curve(human_runs, is_multi, which_isi)
print("ISIs:", isis)
print("Human d' curve:", human_curve)
print(f"Total real sequences: {len(exp_list)}")

/om2/user/bjmedina/auditory-memory/memory/utls/runners_utils.py:210: RuntimeWarning: Mean of empty slice
  dprimes.append(np.nanmean(aucs))


ISIs: [0, 1, 2, 4, 8, 16, 32, 64]
Human d' curve: [3.43265945 3.01600054 2.42799165 2.15901606 2.02618719 1.9135559
 1.79912963 1.60744916]
Total real sequences: 104


## 2. Build encoder & encode stimuli

In [4]:
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = (
    "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,
    model_name=encoder_type,
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=time_avg,
    device="cuda",
)

if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)
X_np = X.detach().cpu().numpy()
print("Encoded shape:", X_np.shape, "  any NaN?", torch.isnan(X).any().item())

Encoder name: resnet50-layer4
LOADING FROM /om2/user/bjmedina/models/cochdnn/model_directories/resnet50_word_speaker_audioset/
=> loading checkpoint '/om2/user/bjmedina/models/cochdnn/model_checkpoints/audio_rep_training_cochleagram_1/standard_training_word_and_audioset_and_speaker_decay_lr/542752d7-9849-49ff-b84a-6758a81585b4/5_checkpoint.pt'
=> loaded checkpoint '/om2/user/bjmedina/models/cochdnn/model_checkpoints/audio_rep_training_cochleagram_1/standard_training_word_and_audioset_and_speaker_decay_lr/542752d7-9849-49ff-b84a-6758a81585b4/5_checkpoint.pt' (epoch 6)
Encoded shape: (80, 186368)   any NaN? False


## 3. Parameter bounds & stimulus pool

In [5]:
def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))


d50 = 1#median_pairwise_distance(X_np, metric="cosine")
print(f"Median pairwise cosine distance: {d50:.6f}")

param_bounds = {
    "sigma0": (
        noise_cfg["sigma0_min"],
        22,
    ),
    "sigma1": (
        0.01,
        10 * d50,
    ),
    "sigma2": (
        noise_cfg["sigma2_min"] * d50,
        0.5 * d50,
    ),
}

print("Parameter bounds:")
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

# Stimulus pool for sequence generation
stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"\nStimulus pool size: {len(stimulus_pool)}")
assert len(stimulus_pool) >= 65, (
    f"Stimulus pool ({len(stimulus_pool)}) too small for ISI-64 blocks (need >= 65)"
)

Median pairwise cosine distance: 1.000000
Parameter bounds:
  sigma0: (3.000000, 22.000000)
  sigma1: (0.010000, 10.000000)
  sigma2: (0.000500, 0.500000)

Stimulus pool size: 80


## 4. Human d' targets

In [6]:
isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}

sigma0_human = {0: float(human_curve[isi_to_hc_idx[0]])}
sigma1_human = {isi: float(human_curve[isi_to_hc_idx[isi]]) for isi in [1, 2, 4]}
sigma2_human = {isi: float(human_curve[isi_to_hc_idx[isi]]) for isi in [8, 16, 32, 64]}

print("Stage A targets (sigma0):")
for isi, dp in sigma0_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

print("\nStage B targets (sigma1):")
for isi, dp in sigma1_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

print("\nStage C targets (sigma2):")
for isi, dp in sigma2_human.items():
    print(f"  ISI {isi}: human d' = {dp:.4f}")

# Initial values for unfitted sigmas
sigma1_init = log_mid(*param_bounds["sigma1"])
sigma2_init = log_mid(*param_bounds["sigma2"])
print(f"\nInitial sigma1 = {sigma1_init:.6f}")
print(f"Initial sigma2 = {sigma2_init:.6f}")

Stage A targets (sigma0):
  ISI 0: human d' = 3.4327

Stage B targets (sigma1):
  ISI 1: human d' = 3.0160
  ISI 2: human d' = 2.4280
  ISI 4: human d' = 2.1590

Stage C targets (sigma2):
  ISI 8: human d' = 2.0262
  ISI 16: human d' = 1.9136
  ISI 32: human d' = 1.7991
  ISI 64: human d' = 1.6074

Initial sigma1 = 0.316228
Initial sigma2 = 0.015811


---
## 5. Custom sweep utilities

A dedicated sweep function that returns **d', AUROC, and per-ISI MSE**
for each sigma1 candidate, plus raw hit/FA score arrays for diagnostics.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

try:
    get_ipython()
    from tqdm.notebook import tqdm, trange
except NameError:
    from tqdm import tqdm, trange


def sweep_sigma1_detailed(
    sigma1_value, sigma0, sigma2, t_step,
    noise_mode, metric, X0, name_to_idx,
    experiment_list, target_isis,
    human_dprimes_by_isi=None,
    n_mc=64, seed=0,
):
    """
    Evaluate a single sigma1 value and return d', AUROC, per-ISI MSE,
    and raw hit/FA scores for diagnostics.
    """
    seq_trial_isis = [infer_trial_isis(seq) for seq in experiment_list]

    all_hits_by_isi = {isi: [] for isi in target_isis}
    all_fas = []

    for rep in range(n_mc):
        for si, seq in enumerate(experiment_list):
            t_isis = seq_trial_isis[si]
            run_out = run_experiment_scores(
                sigma0=sigma0,
                sigma1=sigma1_value,
                sigma2=sigma2,
                t_step=t_step,
                rate=0,
                noise_mode=noise_mode,
                metric=metric,
                X0=X0,
                name_to_idx=name_to_idx,
                experiment_list=[seq],
                debug=False,
                seed=seed + rep * 1000 + si,
            )
            h = np.asarray(run_out["hits"])
            f = np.asarray(run_out["fas"])
            if len(h) != len(t_isis):
                continue
            for isi_val in target_isis:
                mask = np.array(t_isis) == isi_val
                all_hits_by_isi[isi_val].extend(h[mask].tolist())
            all_fas.extend(f.tolist())

    fas_arr = np.array(all_fas)

    result = {
        "sigma1": sigma1_value,
        "dprime_by_isi": {},
        "auroc_by_isi": {},
        "mse_by_isi": {},
        "n_hits_by_isi": {},
        "n_fas": len(fas_arr),
        "hit_scores_by_isi": {},
        "fa_scores": fas_arr,
    }

    for isi_val in target_isis:
        hits_isi = np.array(all_hits_by_isi[isi_val])
        result["n_hits_by_isi"][isi_val] = len(hits_isi)
        result["hit_scores_by_isi"][isi_val] = hits_isi

        if len(hits_isi) == 0 or len(fas_arr) == 0:
            result["dprime_by_isi"][isi_val] = np.nan
            result["auroc_by_isi"][isi_val] = np.nan
            result["mse_by_isi"][isi_val] = np.nan
            continue

        y_true = np.concatenate([np.ones(len(hits_isi)), np.zeros(len(fas_arr))])
        scores = np.concatenate([hits_isi, fas_arr])
        auroc = roc_auc_score(y_true, -scores)
        dprime = auroc_to_dprime(auroc)

        result["auroc_by_isi"][isi_val] = auroc
        result["dprime_by_isi"][isi_val] = dprime

        if human_dprimes_by_isi and isi_val in human_dprimes_by_isi:
            result["mse_by_isi"][isi_val] = (dprime - human_dprimes_by_isi[isi_val]) ** 2
        else:
            result["mse_by_isi"][isi_val] = np.nan

    return result

In [ ]:
# --- sigma1 grid: 30 log-spaced points ---
SIGMA1_GRID = np.exp(np.linspace(np.log(0.01), np.log(10.0), 30))
N_MC = 64

print(f"Sigma1 grid: {len(SIGMA1_GRID)} points, range [{SIGMA1_GRID[0]:.4f}, {SIGMA1_GRID[-1]:.4f}]")


def run_sigma1_sweep(
    sigma0, sigma1_grid, sigma2, t_step,
    experiment_list, target_isis,
    human_dprimes_by_isi,
    noise_mode, metric, X0, name_to_idx,
    n_mc=64, seed=0, desc="sigma1 sweep",
):
    """Run sweep_sigma1_detailed for each sigma1 in the grid."""
    results = []
    for i in trange(len(sigma1_grid), desc=desc):
        r = sweep_sigma1_detailed(
            sigma1_value=sigma1_grid[i],
            sigma0=sigma0,
            sigma2=sigma2,
            t_step=t_step,
            noise_mode=noise_mode,
            metric=metric,
            X0=X0,
            name_to_idx=name_to_idx,
            experiment_list=experiment_list,
            target_isis=target_isis,
            human_dprimes_by_isi=human_dprimes_by_isi,
            n_mc=n_mc,
            seed=seed + i * 100_000,
        )
        results.append(r)
    return results

In [ ]:
def plot_sweep_results(
    sweep_dict, sigma1_grid, human_dprimes_by_isi,
    target_isi=1, title="", label_key="sigma0",
):
    """
    Three-panel figure comparing d', AUROC, and MSE vs sigma1
    across multiple conditions.

    sweep_dict: {label_value: [results_list]}
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    cmap = plt.cm.viridis
    labels = sorted(sweep_dict.keys())
    colors = cmap(np.linspace(0.05, 0.95, len(labels)))

    for color, label in zip(colors, labels):
        results = sweep_dict[label]
        s1_vals = np.array([r["sigma1"] for r in results])
        dp_vals = np.array([r["dprime_by_isi"].get(target_isi, np.nan) for r in results])
        auc_vals = np.array([r["auroc_by_isi"].get(target_isi, np.nan) for r in results])
        mse_vals = np.array([r["mse_by_isi"].get(target_isi, np.nan) for r in results])

        lbl = f"{label_key}={label}"
        axes[0].plot(s1_vals, dp_vals, "o-", ms=3, color=color, label=lbl)
        axes[1].plot(s1_vals, auc_vals, "o-", ms=3, color=color, label=lbl)
        axes[2].plot(s1_vals, mse_vals, "o-", ms=3, color=color, label=lbl)

    # Human d' reference line
    if target_isi in human_dprimes_by_isi:
        axes[0].axhline(
            human_dprimes_by_isi[target_isi], ls=":", color="red",
            alpha=0.7, label=f"human d' (ISI={target_isi})",
        )

    for ax, ylabel in zip(axes, ["d'", "AUROC", "MSE"]):
        ax.set_xscale("log")
        ax.set_xlabel("sigma1")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7, frameon=False)
        ax.grid(alpha=0.25)

    axes[0].set_title(f"d' (ISI={target_isi}) vs sigma1")
    axes[1].set_title(f"AUROC (ISI={target_isi}) vs sigma1")
    axes[2].set_title(f"MSE (ISI={target_isi}) vs sigma1")

    if title:
        fig.suptitle(title, y=1.03, fontsize=13)
    fig.tight_layout()
    plt.show()
    return fig

---
## Investigation A: Effect of sigma0 on ISI-1 non-monotonicity

Sweep sigma1 for many different **fixed sigma0** values.
If the bump only appears for certain sigma0 ranges, that narrows the
hypothesis — sigma0 (encoding noise) may interact with sigma1 (drift noise)
in a way that affects the hit/FA separation at ISI=1.

In [ ]:
# Generate ISI=[1]-only compact sequences
ISI1_EXPS, ISI1_ISI_KEYS = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=[1],
    n_sequences=8,
    length=15,
    min_pairs_per_isi=5,
    seed=1000,
)
print(f"Generated {len(ISI1_EXPS)} ISI=[1] sequences, {len(ISI1_EXPS[0])} trials each")

# Validate
from collections import Counter, defaultdict
for seq in ISI1_EXPS:
    counts = Counter(infer_trial_isis(seq))
    print(f"  ISI=1 pairs: {counts.get(1, 0)}")

In [ ]:
# Sweep sigma1 for a grid of sigma0 values
SIGMA0_VALUES = [3, 5, 8, 10, 13.5, 15, 18, 20, 22]

sweep_by_sigma0 = {}
for s0 in SIGMA0_VALUES:
    print(f"\n--- sigma0 = {s0} ---")
    sweep_by_sigma0[s0] = run_sigma1_sweep(
        sigma0=s0, sigma1_grid=SIGMA1_GRID, sigma2=sigma2_init,
        t_step=t_step, experiment_list=ISI1_EXPS,
        target_isis=[1], human_dprimes_by_isi=sigma1_human,
        noise_mode=noise_mode, metric=metric,
        X0=X, name_to_idx=name_to_idx,
        n_mc=N_MC, seed=200_000,
        desc=f"sigma0={s0}",
    )

In [ ]:
# Plot Investigation A results
fig = plot_sweep_results(
    sweep_by_sigma0, SIGMA1_GRID, sigma1_human,
    target_isi=1,
    title="Investigation A: Effect of sigma0 on d'(ISI=1) vs sigma1",
    label_key="sigma0",
)

---
## Investigation B: Effect of t_step on ISI-1 non-monotonicity

For ISI=1, the repeat arrives at age=2, which falls in the sigma1 regime
for all t_step values tested (4, 5, 6). However, t_step affects drift of
**other memories** in the bank at longer ages, changing the FA distribution.

In [ ]:
# Sweep sigma1 for different t_step values
TSTEP_VALUES = [4, 5, 6]

sweep_by_tstep = {}
for ts in TSTEP_VALUES:
    print(f"\n--- t_step = {ts} ---")
    sweep_by_tstep[ts] = run_sigma1_sweep(
        sigma0=13.5, sigma1_grid=SIGMA1_GRID, sigma2=sigma2_init,
        t_step=ts, experiment_list=ISI1_EXPS,
        target_isis=[1], human_dprimes_by_isi=sigma1_human,
        noise_mode=noise_mode, metric=metric,
        X0=X, name_to_idx=name_to_idx,
        n_mc=N_MC, seed=300_000,
        desc=f"t_step={ts}",
    )

fig = plot_sweep_results(
    sweep_by_tstep, SIGMA1_GRID, sigma1_human,
    target_isi=1,
    title="Investigation B: Effect of t_step on d'(ISI=1) vs sigma1",
    label_key="t_step",
)

---
## Investigation C: Effect of ISI distribution on ISI-1 non-monotonicity

The FA pool differs between ISI=[1]-only sequences (short, 15 trials) and
ISI=[1,2,4] mixed sequences (longer, 60 trials with more diverse foils).
This can change the AUROC/d' calculation and may affect the non-monotonicity.

In [ ]:
# Generate ISI=[1,2,4] compact sequences
ISI124_EXPS, ISI124_ISI_KEYS = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=[1, 2, 4],
    n_sequences=10,
    length=60,
    min_pairs_per_isi=5,
    seed=3000,
)
print(f"Generated {len(ISI124_EXPS)} ISI=[1,2,4] sequences, {len(ISI124_EXPS[0])} trials each")

# Validate ISI distribution
isi_counts = defaultdict(list)
for seq in ISI124_EXPS:
    counts = Counter(infer_trial_isis(seq))
    for isi_val in [1, 2, 4]:
        isi_counts[isi_val].append(counts.get(isi_val, 0))
print("\nPairs per ISI per sequence:")
for isi_val in [1, 2, 4]:
    vals = isi_counts[isi_val]
    print(f"  ISI {isi_val}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}")

# Run sweep with mixed sequences
sweep_isi124 = run_sigma1_sweep(
    sigma0=13.5, sigma1_grid=SIGMA1_GRID, sigma2=sigma2_init,
    t_step=t_step, experiment_list=ISI124_EXPS,
    target_isis=[1, 2, 4], human_dprimes_by_isi=sigma1_human,
    noise_mode=noise_mode, metric=metric,
    X0=X, name_to_idx=name_to_idx,
    n_mc=N_MC, seed=400_000,
    desc="ISI=[1,2,4] sweep",
)

In [ ]:
# Compare ISI=[1]-only vs ISI=[1,2,4] mixed sequences
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reuse ISI=[1]-only results from Investigation A at sigma0=13.5
results_isi1 = sweep_by_sigma0[13.5]
s1_vals = np.array([r["sigma1"] for r in results_isi1])

# Panel 1: d'(ISI=1) comparison
dp_isi1_only = [r["dprime_by_isi"].get(1, np.nan) for r in results_isi1]
dp_isi1_mixed = [r["dprime_by_isi"].get(1, np.nan) for r in sweep_isi124]
axes[0].plot(s1_vals, dp_isi1_only, "o-", ms=3, label="ISI=[1] only")
axes[0].plot(s1_vals, dp_isi1_mixed, "s-", ms=3, label="ISI=[1,2,4] mixed")
axes[0].axhline(sigma1_human[1], ls=":", color="red", alpha=0.7, label="human")
axes[0].set_title("d'(ISI=1) comparison")
axes[0].legend(fontsize=8, frameon=False)

# Panel 2: AUROC(ISI=1) comparison
auc_isi1_only = [r["auroc_by_isi"].get(1, np.nan) for r in results_isi1]
auc_isi1_mixed = [r["auroc_by_isi"].get(1, np.nan) for r in sweep_isi124]
axes[1].plot(s1_vals, auc_isi1_only, "o-", ms=3, label="ISI=[1] only")
axes[1].plot(s1_vals, auc_isi1_mixed, "s-", ms=3, label="ISI=[1,2,4] mixed")
axes[1].set_title("AUROC(ISI=1) comparison")
axes[1].legend(fontsize=8, frameon=False)

# Panel 3: All ISIs from mixed sequences
for isi_val in [1, 2, 4]:
    dp_vals = [r["dprime_by_isi"].get(isi_val, np.nan) for r in sweep_isi124]
    axes[2].plot(s1_vals, dp_vals, "o-", ms=3, label=f"ISI={isi_val}")
    if isi_val in sigma1_human:
        axes[2].axhline(sigma1_human[isi_val], ls=":", alpha=0.5)
axes[2].set_title("d' by ISI (mixed sequences)")
axes[2].legend(fontsize=8, frameon=False)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("sigma1")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("d'")
axes[1].set_ylabel("AUROC")
axes[2].set_ylabel("d'")

fig.suptitle("Investigation C: ISI distribution effect on non-monotonicity", y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## Investigation D: Score distribution diagnostics

Examine raw hit and FA score distributions at three representative sigma1
values (low / bump-peak / high) to understand the mechanism.

If the non-monotonicity comes from FA distances growing faster than hit
distances, we should see the FA distribution shift rightward more rapidly.

In [ ]:
# Pick 3 representative sigma1 values: low, bump-peak, high
# (adjust indices after seeing Investigation A results)
diag_indices = [3, 14, 25]
diag_sigma1s = SIGMA1_GRID[diag_indices]
diag_results = [sweep_by_sigma0[13.5][i] for i in diag_indices]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, (s1, result) in enumerate(zip(diag_sigma1s, diag_results)):
    hits = result["hit_scores_by_isi"][1]
    fas = result["fa_scores"]

    # Top row: overlaid histograms
    ax = axes[0, col]
    lo = min(hits.min(), fas.min())
    hi = max(hits.max(), fas.max())
    bins = np.linspace(lo, hi, 50)
    ax.hist(fas, bins=bins, alpha=0.5, density=True, label="FA", color="tab:blue")
    ax.hist(hits, bins=bins, alpha=0.5, density=True, label=f"Hit (ISI=1)", color="tab:orange")
    ax.set_title(f"sigma1={s1:.4f}")
    ax.legend(fontsize=8)
    ax.set_xlabel("Distance score")
    ax.set_ylabel("Density")

    # Bottom row: ROC curves
    ax = axes[1, col]
    y_true = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
    scores_neg = -np.concatenate([hits, fas])
    fpr, tpr, _ = roc_curve(y_true, scores_neg)
    auroc = roc_auc_score(y_true, scores_neg)
    ax.plot(fpr, tpr, lw=2)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"AUROC={auroc:.4f}, d'={auroc_to_dprime(auroc):.3f}")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")

fig.suptitle("Investigation D: Score distributions at low / bump / high sigma1 (sigma0=13.5)",
             y=1.02, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# How hit and FA distribution statistics change across the sigma1 sweep
results_13 = sweep_by_sigma0[13.5]
s1_vals = np.array([r["sigma1"] for r in results_13])

hit_means = np.array([np.mean(r["hit_scores_by_isi"][1]) for r in results_13])
hit_stds = np.array([np.std(r["hit_scores_by_isi"][1]) for r in results_13])
fa_means = np.array([np.mean(r["fa_scores"]) for r in results_13])
fa_stds = np.array([np.std(r["fa_scores"]) for r in results_13])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Means
axes[0].plot(s1_vals, hit_means, "o-", ms=3, label="Hit mean (ISI=1)", color="tab:orange")
axes[0].plot(s1_vals, fa_means, "s-", ms=3, label="FA mean", color="tab:blue")
axes[0].set_title("Score means vs sigma1")
axes[0].legend(fontsize=8, frameon=False)

# Panel 2: Stds
axes[1].plot(s1_vals, hit_stds, "o-", ms=3, label="Hit std (ISI=1)", color="tab:orange")
axes[1].plot(s1_vals, fa_stds, "s-", ms=3, label="FA std", color="tab:blue")
axes[1].set_title("Score stds vs sigma1")
axes[1].legend(fontsize=8, frameon=False)

# Panel 3: Analytic d' approximation vs actual d'
# For distance metrics, lower = more similar, so hits should have LOWER scores
# d' ~ (mean_FA - mean_Hit) / pooled_std
pooled_std = np.sqrt(0.5 * (hit_stds**2 + fa_stds**2))
analytic_dp = (fa_means - hit_means) / pooled_std
actual_dp = [r["dprime_by_isi"].get(1, np.nan) for r in results_13]

axes[2].plot(s1_vals, analytic_dp, "o-", ms=3, label="Analytic d' approx", color="tab:green")
axes[2].plot(s1_vals, actual_dp, "s-", ms=3, label="Actual d' (AUC-based)", color="tab:red")
axes[2].axhline(sigma1_human[1], ls=":", color="gray", alpha=0.5, label="human target")
axes[2].set_title("d' comparison: analytic vs AUC-based")
axes[2].legend(fontsize=8, frameon=False)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("sigma1")
    ax.grid(alpha=0.25)

fig.suptitle("Investigation D: Hit/FA distribution statistics vs sigma1 (sigma0=13.5)",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()